In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer

In [2]:
# Load dataset
df = pd.read_csv("data/data.csv")
df.head()

,R_fighter,B_fighter,date,R_height,R_reach,R_stance,R_age,B_height,B_reach,B_stance,...,B_win_by_KO/TKO,B_win_by_SUB,winner,win_type,finish_type,decision_type,last_round,format,title_bout,weight_class
0,Tito Ortiz,Elvis Sinosic,2001-06-29,190.50,74.0,Orthodox,26,190.50,77.0,Orthodox,...,0.0,1.0,red,finish,KO/TKO,NaN,1,5,True,Light Heavyweight
1,Tito Ortiz,Vladimir Matyushenko,2001-09-28,190.50,74.0,Orthodox,26,182.88,74.0,Orthodox,...,0.0,0.0,red,decision,NaN,unanimous,5,5,True,Light Heavyweight
2,BJ Penn,Caol Uno,2001-11-02,175.26,70.0,Orthodox,22,170.18,70.0,Southpaw,...,0.5,0.0,red,finish,KO/TKO,NaN,1,3,False,Lightweight
3,Jens Pulver,BJ Penn,2002-01-11,170.18,70.0,Southpaw,27,175.26,70.0,Orthodox,...,1.0,0.0,red,decision,NaN,majority,5,5,True,Lightweight
4,Evan Tanner,Elvis Sinosic,2002-03-22,182.88,74.0,Orthodox,31,190.50,77.0,Orthodox,...,0.0,0.5,red,finish,KO/TKO,NaN,1,3,False,Light Heavyweight


In [3]:
# Convert date column to datetime
df["date"] = pd.to_datetime(df["date"])

# Sort fights so earlier fights come first
df = df.sort_values("date").reset_index(drop=True)

# Convert title_bout to numeric (True/False → 1/0)
df["title_bout"] = df["title_bout"].astype(int)

In [4]:
# Separate Target and Features
# Target label
y1 = df["winner"]
# y = df["winner", "win_type", "finish_type", "decision_type", "last_round"] #all targets

# Drop columns that won't be used as features
X = df.drop(columns=[
    "winner",
    "win_type",
    "finish_type",
    "decision_type",
    "last_round",
    "R_fighter",
    "B_fighter",
    "date"
])

X.shape

(5885, 123)

In [5]:
# Fighter-level KNN imputation (one fighter per row)
# Avoids using opponent-side features directly to fill a fighter's missing stats.

# Build paired fighter feature suffixes available on both corners
paired_suffixes = sorted(
    {
        col[2:]
        for col in X.columns
        if col.startswith("R_") and f"B_{col[2:]}" in X.columns
    }
)

# Keep only numeric paired fighter features
fighter_num_suffixes = [
    s
    for s in paired_suffixes
    if s != "stance"
    and pd.api.types.is_numeric_dtype(X[f"R_{s}"])
    and pd.api.types.is_numeric_dtype(X[f"B_{s}"])
]

# Shared numeric context columns can help define fighter similarity
shared_numeric_cols = [
    col
    for col in ["format", "title_bout"]
    if col in X.columns and pd.api.types.is_numeric_dtype(X[col])
]

row_ids = np.arange(len(X))

red_fighters = pd.DataFrame({"_row_id": row_ids, "_corner": "R"})
blue_fighters = pd.DataFrame({"_row_id": row_ids, "_corner": "B"})

for s in fighter_num_suffixes:
    red_fighters[s] = X[f"R_{s}"].values
    blue_fighters[s] = X[f"B_{s}"].values

for col in shared_numeric_cols:
    red_fighters[col] = X[col].values
    blue_fighters[col] = X[col].values

fighters_long = pd.concat([red_fighters, blue_fighters], ignore_index=True)

# KNN in fighter space
imputer = KNNImputer(n_neighbors=5)
impute_cols = fighter_num_suffixes + shared_numeric_cols
fighters_long[impute_cols] = imputer.fit_transform(fighters_long[impute_cols])

# Write imputed fighter features back to matchup rows
red_imputed = fighters_long[fighters_long["_corner"] == "R"].set_index("_row_id")
blue_imputed = fighters_long[fighters_long["_corner"] == "B"].set_index("_row_id")

for s in fighter_num_suffixes:
    X[f"R_{s}"] = red_imputed.loc[row_ids, s].values
    X[f"B_{s}"] = blue_imputed.loc[row_ids, s].values

X.shape

(5885, 123)

In [6]:
# Categorical columns
cat_cols = ["R_stance", "B_stance", "weight_class"]

# Numeric columns
num_cols = X.columns.drop(cat_cols)

# Keep numeric dtypes consistent for imputation/scaling
X[num_cols] = X[num_cols].astype(float)

# Scale numeric features and encode all categorical columns
transformer = ColumnTransformer(
    [
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop",
)

# Fit & transform the features
X_transformed = transformer.fit_transform(X)

X_transformed.shape

(5885, 141)

In [7]:
# Preparing to train 
# Note: double check for no leakage caused by preceding cells (may need to change order)

import torch
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import TensorDataset, DataLoader

# label encoding converts strings -> integers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y1.values.ravel())
num_classes = len(label_encoder.classes_)

# 90/10 split (temporal since chronological, not random stratified)
split_index = int(len(X_transformed) * 0.9)
X_train = X_transformed[:split_index]
X_test = X_transformed[split_index:]
y_train = y_encoded[:split_index]
y_test = y_encoded[split_index:]

# Temporal split check (date range)
print("\nTrain date range:", df["date"].iloc[:split_index].min(), "to", df["date"].iloc[:split_index].max())
print("Test date range:", df["date"].iloc[split_index:].min(), "to", df["date"].iloc[split_index:].max())


# Converting to tensors
X_train_tensor = torch.tensor(X_train).float()
X_test_tensor = torch.tensor(X_test).float()
y_train_tensor = torch.tensor(y_train).long()
y_test_tensor = torch.tensor(y_test).long()

# DataLoaders
train_ds = TensorDataset(X_train_tensor, y_train_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

print("\nX train tensor shape: ",  X_train_tensor.shape)
print("X test tensor shape: ",  X_test_tensor.shape)
print("num classes: ", num_classes)



Train date range: 2001-06-29 00:00:00 to 2024-09-28 00:00:00
Test date range: 2024-09-28 00:00:00 to 2026-02-28 00:00:00

X train tensor shape:  torch.Size([5296, 141])
X test tensor shape:  torch.Size([589, 141])
num classes:  3


# Winner prediction model is defined at the start of the next cell (training cell)
# so that training always runs with a defined model. Run the next cell to train.

In [9]:
# Winner model (define here so this cell works even if run alone)
input_dim = X_train_tensor.shape[1]

class WinnerClassifier(torch.nn.Module):
    def __init__(self, input_dim, num_classes=3, hidden=128):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_dim, hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden, hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden, num_classes),
        )
    def forward(self, x):
        return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = WinnerClassifier(input_dim, num_classes=num_classes).to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Training loop
num_epochs = 20
model.train()
for epoch in range(num_epochs):
    total_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1}/{num_epochs}, loss: {total_loss / len(train_loader):.4f}")
print("Training done.")

NameError: name 'model' is not defined

In [ ]:
# Evaluate model on test set (accuracy + log loss)
model.eval()
with torch.no_grad():
    logits = model(X_test_tensor.to(device))
    preds = logits.argmax(dim=1).cpu().numpy()
    probs = torch.softmax(logits, dim=1).cpu().numpy()
    test_loss = criterion(logits, y_test_tensor.to(device)).item()

test_acc = (preds == y_test).mean()
# Log loss for multiclass: -mean(log(prob of true class))
eps = 1e-15
log_loss_model = -np.log(probs[np.arange(len(y_test)), y_test] + eps).mean()

print("Model (winner classifier):")
print(f"  Test accuracy:  {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"  Test log loss:  {log_loss_model:.4f}")

In [ ]:
# Random baseline: predict uniformly over classes (red, blue, draw)
np.random.seed(42)
n_test = len(y_test)
random_preds = np.random.randint(0, num_classes, size=n_test)
# Uniform probs => each class gets 1/3
random_probs = np.ones((n_test, num_classes)) / num_classes

random_acc = (random_preds == y_test).mean()
random_log_loss = -np.log(random_probs[np.arange(n_test), y_test] + 1e-15).mean()

print("Random baseline (uniform over 3 classes):")
print(f"  Test accuracy:  {random_acc:.4f} (expected ~{1/3:.4f})")
print(f"  Test log loss:  {random_log_loss:.4f} (expected -log(1/3) = {np.log(3):.4f})")
print()
print("Comparison:")
print(f"  Accuracy:  model {test_acc:.4f}  vs  random {random_acc:.4f}  ->  model wins" if test_acc > random_acc else f"  Accuracy:  model {test_acc:.4f}  vs  random {random_acc:.4f}")
print(f"  Log loss:  model {log_loss_model:.4f}  vs  random {random_log_loss:.4f}  ->  model wins (lower is better)" if log_loss_model < random_log_loss else f"  Log loss:  model {log_loss_model:.4f}  vs  random {random_log_loss:.4f}")